In [ ]:
import sys
from pathlib import Path
carpeta_pipeline = next(ruta for ruta in [Path.cwd(), Path.cwd() / 'pipeline_ingesta', Path.cwd() / 'notebook/unidad_IV/pipeline_ingesta'] if (ruta / 'auditoria_pipeline.py').is_file())
sys.path.insert(0, str(carpeta_pipeline.resolve()))
from auditoria_pipeline import iniciar_etapa

auditoria_etapa = iniciar_etapa('07_publicacion')

# 07 - Auditoria, linaje y publicacion

Ejecutar despues de terminar ambas ramas RAW, profiling, cuarentena y staging, las dimensiones y el paso 6.

Este notebook consulta SQL Server, verifica el balance y las cinco FK, reconcilia medidas y claves, exige auditoria OK de cada etapa y cobertura de linaje por registro.

**Persistencia:** `auditoria.publicaciones`, `auditoria.linaje_ventas`, `auditoria.ejecuciones_etapa` y `auditoria.linaje_registros`. Cada ejecucion tambien deja un log JSONL en `logs/`.

**Publicacion:** `dw.fact_ventas_publicadas` ofrece solo los hechos aprobados por la ultima auditoria. Si la ultima decision es BLOQUEADO, la vista no habilita datos. No se eliminan hechos historicos.

El vinculo directo hecho -> RAW se reconstruye y reconcilia en este paso; las etapas previas registran sus propias relaciones al finalizar. No se inventa una auditoria historica para etapas que nunca se ejecutaron con este registro.

In [ ]:
from conexion_sqlserver import BD_DESTINO

engine_destino = auditoria_etapa.engine
print(f'Auditoria y publicacion en: {BD_DESTINO}')

In [ ]:
from sqlalchemy import text

sql_fuente_linaje = """
WITH cabeceras AS (
    SELECT s.*, r.hash_sha256 AS hash_factura, r.payload AS payload_factura,
        ROW_NUMBER() OVER (
            PARTITION BY s.id_factura ORDER BY s.ultima_edicion DESC, s.id_raw DESC
        ) AS version_cabecera
    FROM staging.invoices s
    INNER JOIN raw.invoices r ON r.id_raw = s.id_raw AND r.lote_id = s.lote_id
    WHERE NOT EXISTS (
        SELECT 1 FROM quarantine.invoices q
        WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
    ) AND NOT EXISTS (
        SELECT 1 FROM raw.validacion_calidad_invoices v
        WHERE v.id_raw = r.id_raw AND v.lote_id = r.lote_id AND v.es_critico = 1
    )
)
SELECT l.id_linea AS id_raw_linea, r.lote_id AS lote_linea,
       r.hash_sha256 AS hash_linea, r.payload AS payload_linea,
       l.id_factura AS numero_factura, l.id_producto AS id_producto_nk,
       l.cantidad, l.neto AS monto_neto, l.impuesto AS monto_impuesto,
       l.total AS monto_total, c.id_cliente AS id_cliente_nk,
       c.id_vendedor AS id_vendedor_nk, c.fecha_factura,
       c.id_raw AS id_raw_factura, c.lote_id AS lote_factura,
       c.hash_factura, c.payload_factura
FROM staging.invoice_lines l
INNER JOIN raw.invoice_lines r ON r.id_raw = l.id_linea
LEFT JOIN cabeceras c ON c.id_factura = l.id_factura AND c.version_cabecera = 1
WHERE NOT EXISTS (
    SELECT 1 FROM quarantine.invoice_lines q
    WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
) AND NOT EXISTS (
    SELECT 1 FROM raw.validacion_calidad v
    WHERE v.id_raw = r.id_raw AND v.lote_id = r.lote_id AND v.es_critico = 1
)
"""

sql_hechos_auditoria = """
SELECT f.*, t.fecha AS fecha_factura,
       c.id_cliente_nk, p.id_producto_nk, v.id_vendedor_nk
FROM dw.fact_ventas f
LEFT JOIN dw.dim_tiempo t ON t.id_tiempo = f.id_tiempo
LEFT JOIN dw.dim_cliente c ON c.id_cliente_sk = f.id_cliente_sk
LEFT JOIN dw.dim_producto p ON p.id_producto_sk = f.id_producto_sk
LEFT JOIN dw.dim_vendedor v ON v.id_vendedor_sk = f.id_vendedor_sk
"""

sql_fk = """
SELECT COUNT(*) FROM dw.fact_ventas f
LEFT JOIN dw.dim_tiempo t ON t.id_tiempo = f.id_tiempo
LEFT JOIN dw.dim_cliente c ON c.id_cliente_sk = f.id_cliente_sk
LEFT JOIN dw.dim_producto p ON p.id_producto_sk = f.id_producto_sk
LEFT JOIN dw.dim_vendedor v ON v.id_vendedor_sk = f.id_vendedor_sk
LEFT JOIN dw.dim_ciudad ci ON ci.id_ciudad_sk = f.id_ciudad_sk
WHERE t.id_tiempo IS NULL OR c.id_cliente_sk IS NULL OR p.id_producto_sk IS NULL
   OR v.id_vendedor_sk IS NULL OR ci.id_ciudad_sk IS NULL
"""

sql_crear_auditoria = """
IF NOT EXISTS (SELECT 1 FROM sys.schemas WHERE name = 'auditoria')
    EXEC('CREATE SCHEMA auditoria');
IF OBJECT_ID('auditoria.publicaciones', 'U') IS NULL
BEGIN
    CREATE TABLE auditoria.publicaciones (
        id_auditoria BIGINT IDENTITY(1,1) PRIMARY KEY,
        id_publicacion NVARCHAR(36) NOT NULL UNIQUE,
        estado NVARCHAR(20) NOT NULL,
        controles_json NVARCHAR(MAX) NOT NULL,
        resumen_json NVARCHAR(MAX) NOT NULL,
        transformaciones_json NVARCHAR(MAX) NOT NULL,
        fecha_auditoria DATETIME2(7) NOT NULL DEFAULT SYSDATETIME()
    );
END;
IF OBJECT_ID('auditoria.linaje_ventas', 'U') IS NULL
BEGIN
    CREATE TABLE auditoria.linaje_ventas (
        id_publicacion NVARCHAR(36) NOT NULL,
        id_fact_venta_sk BIGINT NOT NULL,
        id_raw_linea INT NOT NULL,
        lote_linea NVARCHAR(50) NOT NULL,
        hash_linea NVARCHAR(64) NOT NULL,
        id_raw_factura INT NOT NULL,
        lote_factura NVARCHAR(50) NOT NULL,
        hash_factura NVARCHAR(64) NOT NULL,
        modo_linaje NVARCHAR(30) NOT NULL,
        PRIMARY KEY (id_publicacion, id_fact_venta_sk),
        FOREIGN KEY (id_publicacion) REFERENCES auditoria.publicaciones(id_publicacion),
        FOREIGN KEY (id_fact_venta_sk) REFERENCES dw.fact_ventas(id_fact_venta_sk)
    );
END;
"""

sql_vista_publicada = """
CREATE OR ALTER VIEW dw.fact_ventas_publicadas AS
SELECT f.*
FROM dw.fact_ventas f
INNER JOIN auditoria.linaje_ventas l ON l.id_fact_venta_sk = f.id_fact_venta_sk
INNER JOIN auditoria.publicaciones p ON p.id_publicacion = l.id_publicacion
WHERE p.estado = 'APROBADO'
  AND p.id_auditoria = (SELECT MAX(id_auditoria) FROM auditoria.publicaciones)
"""


def leer_balance(conn):
    filas = []
    for tabla, clave in [('invoice_lines', 'id_linea'), ('invoices', 'id_raw')]:
        sql = f"""
        SELECT (SELECT COUNT(*) FROM raw.{tabla}) AS raw_total,
               (SELECT COUNT(*) FROM staging.{tabla}) AS staging_total,
               (SELECT COUNT(*) FROM raw.{tabla} r WHERE EXISTS (
                   SELECT 1 FROM quarantine.{tabla} q
                   WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
               )) AS cuarentena_total,
               (SELECT COUNT(*) FROM raw.{tabla} r WHERE NOT EXISTS (
                   SELECT 1 FROM staging.{tabla} s WHERE s.{clave} = r.id_raw
               ) AND NOT EXISTS (
                   SELECT 1 FROM quarantine.{tabla} q
                   WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
               )) AS sin_clasificar
        """
        resultado = dict(conn.execute(text(sql)).mappings().one())
        resultado['tabla'] = tabla
        filas.append(resultado)
    return pd.DataFrame(filas)


def contar_rechazados_staging(conn):
    total = 0
    for tabla, validacion, clave in [
        ('invoice_lines', 'validacion_calidad', 'id_linea'),
        ('invoices', 'validacion_calidad_invoices', 'id_raw')
    ]:
        sql = f"""
        SELECT COUNT(*) FROM staging.{tabla} s
        INNER JOIN raw.{tabla} r ON r.id_raw = s.{clave}
        WHERE EXISTS (
            SELECT 1 FROM quarantine.{tabla} q
            WHERE q.id_raw = r.id_raw AND q.lote_id = r.lote_id
        ) OR EXISTS (
            SELECT 1 FROM raw.{validacion} v
            WHERE v.id_raw = r.id_raw AND v.lote_id = r.lote_id AND v.es_critico = 1
        )
        """
        total += conn.execute(text(sql)).scalar_one()
    return total

In [ ]:
import json
from decimal import Decimal
import pandas as pd


def evaluar_publicacion(hechos, fuente, huerfanos, solapamientos):
    fuente = fuente.copy()
    fuente['numero_linea'] = fuente['payload_linea'].map(
        lambda valor: int(json.loads(valor)['InvoiceLineID'])
    )
    fuente['precio_origen'] = fuente['payload_linea'].map(
        lambda valor: Decimal(str(json.loads(valor)['UnitPrice']))
    )
    fuente = fuente.sort_values('id_raw_linea').drop_duplicates('numero_linea', keep='last')
    fuente = fuente.loc[fuente['id_raw_factura'].notna()].copy()
    claves = ['numero_factura', 'numero_linea']
    duplicados = hechos.duplicated(claves).any()
    comparacion = hechos.merge(
        fuente, on=claves, how='left', suffixes=('_dw', '_raw'),
        indicator='_linaje', validate='one_to_one' if not duplicados else 'many_to_one'
    )
    controles = {
        'hay_hechos': not hechos.empty,
        'grano_unico': not bool(duplicados),
        'fk_validas': huerfanos == 0,
        'staging_sin_rechazados': solapamientos == 0,
        'todos_con_origen_aceptado': bool(comparacion['_linaje'].eq('both').all())
    }
    diferencias = 0
    for _, fila in comparacion.loc[comparacion['_linaje'].eq('both')].iterrows():
        pares = [
            ('cantidad_dw', 'cantidad_raw'),
            ('monto_neto_dw', 'monto_neto_raw'),
            ('monto_impuesto_dw', 'monto_impuesto_raw'),
            ('precio_unitario', 'precio_origen')
        ]
        coincide = all(
            pd.notna(fila[destino]) and pd.notna(fila[origen])
            and Decimal(str(fila[destino])) == Decimal(str(fila[origen]))
            for destino, origen in pares
        )
        coincide = coincide and all(
            pd.notna(fila[f'{campo}_dw']) and pd.notna(fila[f'{campo}_raw'])
            and int(fila[f'{campo}_dw']) == int(fila[f'{campo}_raw'])
            for campo in ['id_cliente_nk', 'id_producto_nk', 'id_vendedor_nk']
        )
        coincide = coincide and str(fila['fecha_factura_dw'])[:10] == str(fila['fecha_factura_raw'])[:10]
        diferencias += int(not coincide)
    controles['medidas_y_claves_reconciliadas'] = diferencias == 0
    columnas = [
        'id_fact_venta_sk', 'id_raw_linea', 'lote_linea', 'hash_linea',
        'id_raw_factura', 'lote_factura', 'hash_factura'
    ]
    linaje = comparacion.loc[comparacion['_linaje'].eq('both'), columnas].copy()
    for campo in ['id_fact_venta_sk', 'id_raw_linea', 'id_raw_factura']:
        linaje[campo] = linaje[campo].astype('int64')
    controles['linaje_completo'] = len(linaje) == len(hechos) and not linaje.isna().any().any()
    resumen = {
        'hechos': len(hechos), 'fuentes_aceptadas': len(fuente),
        'hechos_con_linaje': len(linaje), 'diferencias': diferencias,
        'hechos_sin_origen': int(comparacion['_linaje'].ne('both').sum()),
        'aceptadas_sin_hecho': int((~fuente.set_index(claves).index.isin(hechos.set_index(claves).index)).sum())
    }
    return controles, resumen, linaje

## Interpretar correctamente el balance

El balance puede cuadrar y aun asÃ­ **bloquearse** la publicaciÃ³n: una fila
estÃ¡ en cuarentena y el contrato tiene tolerancia cero. Reconciliar no equivale
a aprobar calidad. Las FK y el grano no se verifican aquÃ­: se muestran como
condiciones del caso, no como garantÃ­as obtenidas de una base real.


In [ ]:
import uuid
from auditoria_pipeline import verificar_cobertura

TRANSFORMACIONES = {
    'monto_neto': 'staging.invoice_lines.neto = ExtendedPrice - TaxAmount',
    'monto_impuesto': 'staging.invoice_lines.impuesto = TaxAmount',
    'precio_unitario': 'raw.invoice_lines.payload.UnitPrice',
    'numero_linea': 'raw.invoice_lines.payload.InvoiceLineID',
    'cliente_producto': 'NK -> SK por vigencia SCD2; fallback actual informado en paso 6',
    'vendedor': 'SalespersonPersonID -> dw.dim_vendedor',
    'ciudad': 'geografia de la version resuelta del cliente -> dw.dim_ciudad',
    'margen_costo': 'LineProfit y neto - LineProfit; NULL cuando no existe LineProfit'
}


def guardar_publicacion(conn, id_publicacion, controles, resumen, linaje):
    estado = 'APROBADO' if all(controles.values()) else 'BLOQUEADO'
    conn.execute(text('''
        INSERT INTO auditoria.publicaciones
            (id_publicacion, estado, controles_json, resumen_json, transformaciones_json)
        VALUES (:id, :estado, :controles, :resumen, :transformaciones)
    '''), {
        'id': id_publicacion, 'estado': estado,
        'controles': json.dumps(controles), 'resumen': json.dumps(resumen, default=str),
        'transformaciones': json.dumps(TRANSFORMACIONES)
    })
    if estado == 'APROBADO' and not linaje.empty:
        registros = linaje.copy()
        registros['id_publicacion'] = id_publicacion
        registros['modo_linaje'] = 'RECONSTRUIDO_VERIFICADO'
        registros.to_sql('linaje_ventas', conn, schema='auditoria',
                         if_exists='append', index=False, chunksize=500)
    return estado


ID_PUBLICACION = str(uuid.uuid4())
with engine_destino.begin() as conn:
    conn.execute(text(sql_crear_auditoria))
    hechos = pd.read_sql_query(text(sql_hechos_auditoria), conn, coerce_float=False)
    fuente = pd.read_sql_query(text(sql_fuente_linaje), conn, coerce_float=False)
    huerfanos = conn.execute(text(sql_fk)).scalar_one()
    solapamientos = contar_rechazados_staging(conn)
    balance = leer_balance(conn)
    controles, resumen, linaje_publicado = evaluar_publicacion(
        hechos, fuente, huerfanos, solapamientos
    )
    controles['raw_clasificado'] = bool(balance['sin_clasificar'].eq(0).all())
    controles['aceptados_con_hecho'] = resumen['aceptadas_sin_hecho'] == 0
    controles['balance_filas'] = bool((
        balance['raw_total'] == balance['staging_total'] + balance['cuarentena_total']
    ).all())
    controles['auditoria_y_linaje_por_etapa'], cobertura = verificar_cobertura(
        conn, linaje_publicado, hechos
    )
    resumen['cobertura_auditoria'] = cobertura
    resumen['balance'] = balance.to_dict('records')
    ESTADO_PUBLICACION = guardar_publicacion(
        conn, ID_PUBLICACION, controles, resumen, linaje_publicado
    )
    conn.execute(text(sql_vista_publicada))

display(balance)
display(pd.DataFrame(controles.items(), columns=['control', 'cumple']))
print(f'Publicacion {ID_PUBLICACION}: {ESTADO_PUBLICACION}')
if ESTADO_PUBLICACION != 'APROBADO':
    raise RuntimeError('Publicacion bloqueada y registrada en auditoria.publicaciones; revisar controles')
display(linaje_publicado.head(20))
print('[OK] Datos habilitados en dw.fact_ventas_publicadas con linaje a ambas fuentes RAW')

## Actividad de discusiÃ³n

Â¿Por quÃ© reconciliar contra el origen aceptado y tambiÃ©n informar lo rechazado? **Respuesta:** comparar solo contra el subconjunto aceptado podrÃ­a ocultar el impacto de las exclusiones sobre el total del negocio.

In [ ]:
auditoria_etapa.finalizar(globals())